# Paper Replication (HI-Small)

This notebook tests a paper-aligned replication of the HI-Small **GFP + XGBoost** baseline using a 60/20/20 temporal split and imbalance-aware training. It builds past-only streaming graph-inspired features as a practical proxy for GFP, then tunes and evaluates XGBoost with minority-class metrics. The main output compares achieved HI-Small F1 against the paper’s reported reference.

In [1]:
import json
import math
import os
import platform
import sys
import time
from collections import Counter, defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

# Colab deps (safe no-op outside Colab)
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %pip -q install xgboost scikit-learn

import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

print('Python:', platform.python_version())
print('XGBoost:', xgb.__version__)

PROJECT_ROOT = Path('/content/drive/MyDrive/math-168') if IS_COLAB else Path('/Users/sophiasharif/projects/math-168')
DATA_DIR = PROJECT_ROOT / 'data'
TRANS_CSV = DATA_DIR / 'HI-Small_Trans.csv'
ACCOUNTS_CSV = DATA_DIR / 'HI-Small_accounts.csv'

if not TRANS_CSV.exists() or not ACCOUNTS_CSV.exists():
    raise FileNotFoundError(f'Missing HI-Small files in {DATA_DIR}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.12
XGBoost: 3.2.0
PROJECT_ROOT: /content/drive/MyDrive/math-168
DATA_DIR: /content/drive/MyDrive/math-168/data


In [3]:
# Sanity-check baseline: logistic regression
# Kept for orientation only; paper target is GFP + XGBoost.
def best_f1_threshold(y_true, p):
    ts = np.arange(0.01, 0.99, 0.01)
    best_t, best_f1 = 0.5, -1.0
    for t in ts:
        f = f1_score(y_true, (p >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, float(t)
    return best_t, float(best_f1)

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train)
Xva = scaler.transform(X_val)
Xte = scaler.transform(X_test)

pos = max(1, int(y_train.sum()))
neg = int(len(y_train) - pos)
class_weight = {0: 1.0, 1: neg / pos}

lr = LogisticRegression(max_iter=200, class_weight=class_weight, n_jobs=-1)
lr.fit(Xtr, y_train)
p_val = lr.predict_proba(Xva)[:, 1]
p_test = lr.predict_proba(Xte)[:, 1]

t_lr, f1_lr_val_best = best_f1_threshold(y_val, p_val)
pred_05 = (p_test >= 0.5).astype(int)
pred_best = (p_test >= t_lr).astype(int)

sanity_metrics = {
    'model': 'logistic_regression',
    'val_best_f1': float(f1_lr_val_best),
    'val_best_threshold': float(t_lr),
    'test_f1_at_0.5': float(f1_score(y_test, pred_05, zero_division=0)),
    'test_f1_at_best_threshold': float(f1_score(y_test, pred_best, zero_division=0)),
    'test_precision_at_0.5': float(precision_score(y_test, pred_05, zero_division=0)),
    'test_recall_at_0.5': float(recall_score(y_test, pred_05, zero_division=0)),
    'test_pr_auc': float(average_precision_score(y_test, p_test)),
}
print(json.dumps(sanity_metrics, indent=2))

: 

: 

In [ ]:
# Paper-style replication attempt: GFP + XGBoost (HI-Small Table 2 target)
# Paper target: minority-class F1 at threshold 0.5, reported as mean +- std across seeds.

def minority_metrics_at_05(y_true, p):
    pred = (p >= 0.5).astype(int)
    return {
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'pr_auc': float(average_precision_score(y_true, p)),
    }

def sample_xgb_params(rng):
    # Appendix C/Table 10 ranges for XGBoost
    return {
        'n_estimators': int(rng.integers(10, 1001)),
        'max_depth': int(rng.integers(1, 16)),
        'learning_rate': loguniform(-2.5, -1.0, rng),
        'reg_lambda': loguniform(-2.0, 2.0, rng),
        'scale_pos_weight': float(rng.uniform(1.0, 10.0)),
        'colsample_bytree': float(rng.uniform(0.5, 1.0)),
        'subsample': float(rng.uniform(0.5, 1.0)),
    }

def train_eval_xgb(params, seed, Xtr, ytr, Xva, yva):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        tree_method='hist',
        random_state=int(seed),
        n_jobs=-1,
        **params,
    )
    model.fit(Xtr, ytr, verbose=False)
    p_val = model.predict_proba(Xva)[:, 1]
    return minority_metrics_at_05(yva, p_val), model

def successive_halving_xgb(Xtr, ytr, Xva, yva, x0, eta, r0, seed=42):
    rng = np.random.default_rng(seed)
    candidates = [sample_xgb_params(rng) for _ in range(x0)]

    round_idx = 0
    frac = float(r0)
    history = []

    while len(candidates) > 1 and frac <= 1.000001:
        n_use = max(1000, int(len(Xtr) * min(frac, 1.0)))
        X_sub, y_sub = Xtr[:n_use], ytr[:n_use]

        scored = []
        for i, params in enumerate(candidates):
            metrics, _ = train_eval_xgb(params, seed + round_idx + i, X_sub, y_sub, Xva, yva)
            scored.append((metrics['f1'], params))

        scored.sort(key=lambda z: z[0], reverse=True)
        keep = max(1, int(math.ceil(len(scored) / eta)))
        candidates = [p for _, p in scored[:keep]]

        history.append({
            'round': round_idx,
            'train_fraction': min(frac, 1.0),
            'num_candidates_in': len(scored),
            'num_candidates_out': len(candidates),
            'best_val_f1_at_0.5': float(scored[0][0]),
        })

        frac *= eta
        round_idx += 1

        if frac > 1.0 and len(candidates) > 1:
            frac = 1.0

    best_params = candidates[0]
    return best_params, history

# Paper Table 9 (small, multi-bank): x0=1000, eta=2, r0=0.1
# For practical local runtime, defaults are reduced but configurable.
x0 = int(os.getenv('PAPER_REPL_X0', '64'))
eta = float(os.getenv('PAPER_REPL_ETA', '2'))
r0 = float(os.getenv('PAPER_REPL_R0', '0.1'))

t0 = time.time()
best_params, sh_history = successive_halving_xgb(
    X_train,
    y_train,
    X_val,
    y_val,
    x0=x0,
    eta=eta,
    r0=r0,
    seed=42,
)
print('successive-halving seconds:', round(time.time() - t0, 2))
print('best params:', json.dumps(best_params, indent=2))
print('history:', json.dumps(sh_history, indent=2))

# Paper reports mean +- std across multiple random seeds.
seed_list = [11, 22, 33, 44]
seed_results = []

for s in seed_list:
    metrics_val, model = train_eval_xgb(best_params, s, X_train, y_train, X_val, y_val)
    p_test = model.predict_proba(X_test)[:, 1]
    metrics_test = minority_metrics_at_05(y_test, p_test)
    seed_results.append({
        'seed': int(s),
        'val_f1_at_0.5': metrics_val['f1'],
        'test_f1_at_0.5': metrics_test['f1'],
        'test_precision_at_0.5': metrics_test['precision'],
        'test_recall_at_0.5': metrics_test['recall'],
        'test_pr_auc': metrics_test['pr_auc'],
    })

test_f1s = np.array([r['test_f1_at_0.5'] for r in seed_results], dtype=np.float64)
test_precs = np.array([r['test_precision_at_0.5'] for r in seed_results], dtype=np.float64)
test_recs = np.array([r['test_recall_at_0.5'] for r in seed_results], dtype=np.float64)
test_ap = np.array([r['test_pr_auc'] for r in seed_results], dtype=np.float64)

paper_hi_small_reference = 0.6323

paper_style_metrics = {
    'model_target_from_paper': 'GFP + XGBoost (Table 2, HI-Small)',
    'paper_hi_small_f1_reference_at_0.5': float(paper_hi_small_reference),
    'replication_test_f1_mean_at_0.5': float(test_f1s.mean()),
    'replication_test_f1_std_at_0.5': float(test_f1s.std(ddof=1) if len(test_f1s) > 1 else 0.0),
    'replication_test_precision_mean_at_0.5': float(test_precs.mean()),
    'replication_test_recall_mean_at_0.5': float(test_recs.mean()),
    'replication_test_pr_auc_mean': float(test_ap.mean()),
    'gap_vs_paper_f1': float(test_f1s.mean() - paper_hi_small_reference),
    'rows_used': int(n),
    'train_pos': int(y_train.sum()),
    'val_pos': int(y_val.sum()),
    'test_pos': int(y_test.sum()),
    'search_config_used': {'x0': x0, 'eta': eta, 'r0': r0},
    'paper_small_search_reference': {'x0': 1000, 'eta': 2, 'r0': 0.1},
    'best_params': best_params,
    'seed_results': seed_results,
}
print(json.dumps(paper_style_metrics, indent=2))

artifacts = PROJECT_ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)
with open(artifacts / 'sanity_logreg_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(sanity_metrics, f, indent=2)
with open(artifacts / 'paper_style_xgb_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(paper_style_metrics, f, indent=2)

# Optional: compare to motif-GNN notebook output if present.
motif_file = artifacts / 'motif_gnn_quick_metrics.json'
if motif_file.exists():
    motif = json.loads(motif_file.read_text())
    print('\nMotif-GNN comparison (from model.ipynb):')
    print(json.dumps({
        'motif_test_f1': motif.get('test_f1'),
        'xgb_test_f1_mean_at_0.5': paper_style_metrics['replication_test_f1_mean_at_0.5'],
        'logreg_test_f1_at_0.5': sanity_metrics['test_f1_at_0.5'],
    }, indent=2))

print('\nSaved metrics to artifacts/: sanity_logreg_metrics.json, paper_style_xgb_metrics.json')

{
  "model": "xgboost_paper_style",
  "paper_hi_small_best_f1_reference": 0.6323,
  "val_best_f1": 0.033251833740831294,
  "threshold": 0.93,
  "test_f1": 0.0350597609561753,
  "test_precision": 0.020813623462630087,
  "test_recall": 0.1111111111111111,
  "test_pr_auc": 0.015246161371081412,
  "rows_used": 2000000,
  "train_pos": 361,
  "val_pos": 227,
  "test_pos": 396
}

Motif-GNN comparison (from model.ipynb):
{
  "motif_test_f1": 0.0,
  "xgb_test_f1": 0.0350597609561753,
  "logreg_test_f1": 0.009307033352260104
}

Saved metrics to artifacts/: sanity_logreg_metrics.json, paper_style_xgb_metrics.json
